# Development data reconciliation and sealed calendar checks

This notebook reruns the development data reconciliation and reads the existing date-only coverage audit. It does not read holdout ZIPs or price CSVs, calculate returns or states, or score models. Prices are compared as exact decimal values; rounding is not used to conceal differences.

## Methods and boundaries

Inputs are limited to the three `*_1d_2020-01_2023-12.csv` files and the mentor's CSV. The reconciliation module validates dates before converting numerical values and rejects rows from 2024 onward. Its source is `tools/reconcile_development.py`; download coverage is audited by `tools/audit_download_coverage.py`. Original field names are retained below for traceability.


In [1]:
from pathlib import Path
import json, sys, html
from IPython.display import display, HTML
ROOT = Path.cwd() if (Path.cwd() / 'tools').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from tools.reconcile_development import AuditConfig, reconcile
mentor = ROOT.parent.parent / 'ChatGPT' / 'bitcoin' / 'CTA_public' / 'AFP_binance_spots.csv'
config = AuditConfig(ROOT / 'data/development', mentor, ROOT / 'results/data_audit')
status, report, differences = reconcile(config)

def show_table(rows):
    if not rows:
        print('No rows'); return
    columns = list(rows[0])
    header = ''.join('<th>' + html.escape(str(x)) + '</th>' for x in columns)
    body = ''.join('<tr>' + ''.join('<td>' + html.escape(str(r[c])) + '</td>' for c in columns) + '</tr>' for r in rows)
    display(HTML('<table><thead><tr>' + header + '</tr></thead><tbody>' + body + '</tbody></table>'))


## Date and closing-price agreement

First confirm that each comparison concerns the same asset and date. Duplicate keys are not resolved by arbitrarily retaining the first or last row. Calendar coverage and numerical agreement are reported separately.


In [2]:
rows = []
for asset, result in report['assets'].items():
    coverage = result['coverage']['development']
    close = result['fields']['close']
    rows.append({'asset': asset, 'start': coverage['first_date'], 'end': coverage['last_date'],
                 'rows': coverage['rows'], 'common_dates': result['compared_unique_dates'],
                 'close_exact_matches': close['exact_matches'],
                 'duplicate_dates': len(coverage['duplicates']),
                 'internal_gaps': len(coverage['gaps_within_observed_span'])})
show_table(rows)
print('All columns exactly reconciled:', report['exact_reconciliation'])


asset,start,end,rows,common_dates,close_exact_matches,duplicate_dates,internal_gaps
BTCUSDT,2020-01-01,2023-12-31,1461,1461,1461,0,0
ETHUSDT,2020-01-01,2023-12-31,1461,1461,1461,0,0
SOLUSDT,2020-08-11,2023-12-31,1238,1238,1238,0,0


All columns exactly reconciled: False


## Differences in other fields

The table below lists every field with an exact mismatch. Fixed tolerances describe the size of differences; a value within tolerance is not relabeled as exactly equal. A tool exit status of 1 can therefore indicate a faithfully reported discrepancy rather than a program crash.


In [3]:
field_rows = []
for asset, result in report['assets'].items():
    for field, item in result['fields'].items():
        if item['exact_mismatches']:
            field_rows.append({'asset': asset, 'field': field,
                               'nonidentical_dates': item['exact_mismatches'],
                               'outside_fixed_tolerance': item['outside_fixed_tolerance'],
                               'max_absolute_difference': item['maximum_absolute_difference']})
show_table(field_rows)


asset,field,nonidentical_dates,outside_fixed_tolerance,max_absolute_difference
BTCUSDT,volume,68,46,3200.89373200
BTCUSDT,quote_volume,1113,46,105563012.56879608
BTCUSDT,taker_buy_quote,839,46,47850695.40643282
ETHUSDT,volume,73,47,56043.76933000
ETHUSDT,quote_volume,248,47,70276390.79616220
ETHUSDT,taker_buy_quote,153,47,32962866.81853000
SOLUSDT,open,6,6,0.01040000
SOLUSDT,volume,63,46,191522.86000000
SOLUSDT,quote_volume,64,46,12499422.07260000
SOLUSDT,taker_buy_quote,72,46,6455785.04980000


## Date-only checks for the new holdout

The following inputs are JSON files produced by the downloader and coverage tool. They contain no prices. An incomplete tail must remain marked incomplete; the test endpoint must not be changed for convenience. The coverage audit separately records the basis for excluding SOL dates before listing.


In [4]:
coverage_path = ROOT / 'results/data_audit/download_coverage_audit.json'
coverage_audit = json.loads(coverage_path.read_text())
# Show six bounded, price-free aggregate rows; archive receipts remain separate.
coverage_rows = []
for partition, part in coverage_audit['partitions'].items():
    for asset, entry in part['symbols'].items():
        coverage_rows.append({'partition': partition, 'asset': asset,
                              'rows': entry['observed_rows'],
                              'first': entry['first_observed_date'],
                              'last': entry['last_observed_date'],
                              'missing_eligible_days': len(entry['missing_eligible_dates']),
                              'interior_gaps': len(entry['interior_missing_dates']),
                              'complete': entry['eligible_requested_range_complete']})
show_table(coverage_rows)
print('Target remains 2026-09-30; available prefix is not a completed holdout.')


partition,asset,rows,first,last,missing_eligible_days,interior_gaps,complete
development,BTCUSDT,1461,2020-01-01,2023-12-31,0,0,True
development,ETHUSDT,1461,2020-01-01,2023-12-31,0,0,True
development,SOLUSDT,1238,2020-08-11,2023-12-31,0,0,True
holdout,BTCUSDT,974,2024-01-01,2026-08-31,30,0,False
holdout,ETHUSDT,974,2024-01-01,2026-08-31,30,0,False
holdout,SOLUSDT,974,2024-01-01,2026-08-31,30,0,False


Target remains 2026-09-30; available prefix is not a completed holdout.


## Saved input hashes

The reconciliation hashes help detect changes to inputs. They do not replace sound methodology or prove that nobody inspected the data.


In [5]:
show_table([{'source': name, 'bytes': info['bytes'], 'rows': info['rows'],
             'sha256': info['sha256']} for name, info in report['inputs'].items()])


source,bytes,rows,sha256
BTCUSDT,215751,1461,64ae4f3a75f61e216c36a7cb41eacaeb645ce3c27b6fd1e4ad3da9c87ef957f5
ETHUSDT,210173,1461,b515b0bc0470129317caa7e0a1e79f6edd4140662921e0f2a10edbdec5488b44
SOLUSDT,168734,1238,2eed2fd031bac2a970dc5fdadbdbe774c6835bd2eac8fed15f69107a7747f792
mentor,744849,4160,49db1e268d1526aa8ae54711fe60e7867b0fddcbf1aaccd30cbc300b2cc94152


## Interpretation limits

In this executed audit, development dates and closing prices agree for all three assets, while some opening prices and volume fields differ. The predefined main model uses closing prices only, so that work can continue. Research using volume requires a separate investigation of the discrepancies. This reconciliation does not establish structural assumptions, predictive ability, availability of the unpublished holdout tail, or protocol freeze status.

See `research/data_audit_2026_10_02.md` for details. HTML was exported from the executed results; browser layout has not been visually reviewed.
